## Dataset Preparation

In [ ]:
# Install the packages required by this notebook in a fresh Colab runtime.
# %pip targets the active notebook kernel, which avoids installing into a
# different Python environment.
%pip install -q tensorflow scikit-learn matplotlib scipy pandas

print("Required packages installed.")

In [ ]:
import tensorflow as tf
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Human-readable CIFAR10 labels used in ROC-curve legends.
cifar10_class_names = [
    'airplane',
    'automobile',
    'bird',
    'cat',
    'deer',
    'dog',
    'frog',
    'horse',
    'ship',
    'truck'
]

print("CIFAR10 class names defined.")

### Load and preprocess CIFAR10 and MNIST datasets

In [ ]:
import tensorflow as tf
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Load CIFAR10 dataset
# cifar_x_train, cifar_y_train: Training images and labels (50000, 32, 32, 3) and (50000, 1)
# cifar_x_test, cifar_y_test: Test images and labels (10000, 32, 32, 3) and (10000, 1)
(cifar_x_train, cifar_y_train), (cifar_x_test, cifar_y_test) = tf.keras.datasets.cifar10.load_data()

# Combine train and test for CIFAR10 for consistent preprocessing and splitting later.
# This ensures that the entire dataset is treated uniformly before splitting into
# new train, validation, and test sets. Resulting shape: (60000, 32, 32, 3) and (60000, 1)
cifar_x = np.concatenate((cifar_x_train, cifar_x_test), axis=0)
cifar_y = np.concatenate((cifar_y_train, cifar_y_test), axis=0)

# Load MNIST dataset
# mnist_x_train, mnist_y_train: Training images and labels (60000, 28, 28) and (60000,)
# mnist_x_test, mnist_y_test: Test images and labels (10000, 28, 28) and (10000,)
(mnist_x_train, mnist_y_train), (mnist_x_test, mnist_y_test) = tf.keras.datasets.mnist.load_data()

# Combine train and test for MNIST for consistent preprocessing and splitting later.
# Similar to CIFAR10, this creates a single pool of MNIST data. Resulting shape: (70000, 28, 28) and (70000,)
mnist_x = np.concatenate((mnist_x_train, mnist_x_test), axis=0)
mnist_y = np.concatenate((mnist_y_train, mnist_y_test), axis=0)

print(f"CIFAR10 original shape: {cifar_x.shape}, {cifar_y.shape}")
print(f"MNIST original shape: {mnist_x.shape}, {mnist_y.shape}")

In [ ]:
import tensorflow as tf
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Convert CIFAR10 to grayscale
# The CIFAR10 images are 3-channel (RGB). For consistency with MNIST (grayscale), convert them to grayscale.
# Formula used for grayscale conversion: 0.2989 * R + 0.5870 * G + 0.1140 * B
# The output `cifar_x_gray` will be 3-dimensional (height, width, 1 channel).
cifar_x_gray = np.dot(cifar_x[...,:3], [0.2989, 0.5870, 0.1140])
cifar_x_gray = np.expand_dims(cifar_x_gray, axis=-1) # Add channel dimension (e.g., (60000, 32, 32) -> (60000, 32, 32, 1))

print(f"CIFAR10 grayscale shape: {cifar_x_gray.shape}")

# MNIST is already grayscale (single channel), but ensure it has an explicit channel dimension for consistency
# with CNN input expectations (e.g., (70000, 28, 28) -> (70000, 28, 28, 1)).
mnist_x_gray = np.expand_dims(mnist_x, axis=-1)

print(f"MNIST with channel dimension shape: {mnist_x_gray.shape}")

In [ ]:
import tensorflow as tf
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Rescale CIFAR10 images to 28x28 pixels to match MNIST dimensions.
# tf.image.resize is used with bilinear interpolation to smooth the resizing process.
# The output is converted to a NumPy array for further processing.
cifar_x_resized = tf.image.resize(cifar_x_gray, [28, 28], method=tf.image.ResizeMethod.BILINEAR).numpy()

print(f"CIFAR10 resized shape: {cifar_x_resized.shape}")
# MNIST images are already 28x28, so no resizing is needed for consistency.
print(f"MNIST (no resize needed) shape: {mnist_x_gray.shape}")

In [ ]:
import tensorflow as tf
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Normalize intensity levels to a specific range (50-200)
# This function takes image data (assumed to be 0-255) and scales it to the target range.
# The formula `(images / 255.0) * (target_max - target_min) + target_min` is used.
def normalize_intensity(images):
    # Scale original 0-255 to target 50-200
    return (images / 255.0) * (200 - 50) + 50

cifar_x_norm = normalize_intensity(cifar_x_resized)
mnist_x_norm = normalize_intensity(mnist_x_gray)

print(f"CIFAR10 min intensity: {np.min(cifar_x_norm)}, max intensity: {np.max(cifar_x_norm)}")
print(f"MNIST min intensity: {np.min(mnist_x_norm)}, max intensity: {np.max(mnist_x_norm)}")

### Split the datasets into training (70%), validation (20%), and test (10%) sets

In [ ]:
import tensorflow as tf
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# For CIFAR10, split the normalized data into training (70%), validation (20%), and test (10%) sets.
# First split: 70% train, 30% temporary (for val/test)
X_cifar_train, X_cifar_temp, y_cifar_train, y_cifar_temp = train_test_split(
    cifar_x_norm, cifar_y, test_size=0.3, random_state=42)

# Second split: From the 30% temporary set, split into 2/3 for validation (20% of total) and 1/3 for test (10% of total)
# test_size=1/3 of the temporary set results in 0.1 / 0.3 = 1/3
X_cifar_val, X_cifar_test, y_cifar_val, y_cifar_test = train_test_split(
    X_cifar_temp, y_cifar_temp, test_size=1/3, random_state=42)

print(f"CIFAR10 Train set shape: {X_cifar_train.shape}, {y_cifar_train.shape}")
print(f"CIFAR10 Validation set shape: {X_cifar_val.shape}, {y_cifar_val.shape}")
print(f"CIFAR10 Test set shape: {X_cifar_test.shape}, {y_cifar_test.shape}")

# For MNIST, apply the same splitting logic for consistency.
# First split: 70% train, 30% temporary (for val/test)
X_mnist_train, X_mnist_temp, y_mnist_train, y_mnist_temp = train_test_split(
    mnist_x_norm, mnist_y, test_size=0.3, random_state=42)

# Second split: From the 30% temporary set, split into 2/3 for validation (20% of total) and 1/3 for test (10% of total)
X_mnist_val, X_mnist_test, y_mnist_val, y_mnist_test = train_test_split(
    X_mnist_temp, y_mnist_temp, test_size=1/3, random_state=42)

print(f"MNIST Train set shape: {X_mnist_train.shape}, {y_mnist_train.shape}")
print(f"MNIST Validation set shape: {X_mnist_val.shape}, {y_mnist_val.shape}")
print(f"MNIST Test set shape: {X_mnist_test.shape}, {y_mnist_test.shape}")

## Task 1: PCA-based Representation Learning

### Standard PCA

In [ ]:
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

# Flatten and Mean-Center CIFAR10 training data
X_cifar_train_flat = X_cifar_train.reshape(X_cifar_train.shape[0], -1)
mean_cifar = np.mean(X_cifar_train_flat, axis=0)
X_cifar_train_centered = X_cifar_train_flat - mean_cifar

# Flatten and Mean-Center MNIST training data
X_mnist_train_flat = X_mnist_train.reshape(X_mnist_train.shape[0], -1)
mean_mnist = np.mean(X_mnist_train_flat, axis=0)
X_mnist_train_centered = X_mnist_train_flat - mean_mnist

print(f"CIFAR10 training data (flattened and centered) shape: {X_cifar_train_centered.shape}")
print(f"MNIST training data (flattened and centered) shape: {X_mnist_train_centered.shape}")

In [ ]:
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

# Apply Standard PCA for CIFAR10
# Initialize PCA with 30 components and a random state for reproducibility.
# `n_components=30` means we reduce the dimensionality from 784 (28x28) to 30.
pca_cifar = PCA(n_components=30, random_state=42)

# Fit PCA on the mean-centered CIFAR10 training data and transform it.
# `fit_transform` learns the principal components and projects the data onto them.
X_cifar_train_pca = pca_cifar.fit_transform(X_cifar_train_centered)

# Transform CIFAR10 validation and test data using the PCA model fitted on the training data.
# It's crucial to use the mean and components learned from the training set to avoid data leakage.
# Flatten the validation data, mean-center it using the mean from the training set, then transform.
X_cifar_val_flat = X_cifar_val.reshape(X_cifar_val.shape[0], -1)
X_cifar_val_centered = X_cifar_val_flat - mean_cifar
X_cifar_val_pca = pca_cifar.transform(X_cifar_val_centered)

# Flatten the test data, mean-center it using the mean from the training set, then transform.
X_cifar_test_flat = X_cifar_test.reshape(X_cifar_test.shape[0], -1)
X_cifar_test_centered = X_cifar_test_flat - mean_cifar
X_cifar_test_pca = pca_cifar.transform(X_cifar_test_centered)

print(f"CIFAR10 PCA training features shape: {X_cifar_train_pca.shape}")
print(f"CIFAR10 PCA validation features shape: {X_cifar_val_pca.shape}")
print(f"CIFAR10 PCA test features shape: {X_cifar_test_pca.shape}")

# Apply Standard PCA for MNIST
# Initialize PCA for MNIST with the same number of components and random state.
pca_mnist = PCA(n_components=30, random_state=42)

# Fit PCA on the mean-centered MNIST training data and transform it.
X_mnist_train_pca = pca_mnist.fit_transform(X_mnist_train_centered)

# Transform MNIST validation and test data using the PCA model fitted on the training data.
# Flatten the validation data, mean-center it using the mean from the training set, then transform.
X_mnist_val_flat = X_mnist_val.reshape(X_mnist_val.shape[0], -1)
X_mnist_val_centered = X_mnist_val_flat - mean_mnist
X_mnist_val_pca = pca_mnist.transform(X_mnist_val_centered)

# Flatten the test data, mean-center it using the mean from the training set, then transform.
X_mnist_test_flat = X_mnist_test.reshape(X_mnist_test.shape[0], -1)
X_mnist_test_centered = X_mnist_test_flat - mean_mnist
X_mnist_test_pca = pca_mnist.transform(X_mnist_test_centered)

print(f"MNIST PCA training features shape: {X_mnist_train_pca.shape}")
print(f"MNIST PCA validation features shape: {X_mnist_val_pca.shape}")
print(f"MNIST PCA test features shape: {X_mnist_test_pca.shape}")

### Train Logistic Regression Classifiers using PCA features

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

# NOTE: X_cifar_train_centered and X_mnist_train_centered are already defined
# in a previous cell (fb02101c) to resolve NameError.

# Train Logistic Regression for CIFAR10
# Initialize Logistic Regression model with an increased max_iter for convergence.
# random_state ensures reproducibility.
logistic_cifar = LogisticRegression(max_iter=10000, random_state=42)
# Fit the model using CIFAR10 PCA training features and corresponding labels.
logistic_cifar.fit(X_cifar_train_pca, y_cifar_train.ravel())

# Evaluate CIFAR10 classifier
# Calculate accuracy on the PCA-transformed CIFAR10 test set.
accuracy_cifar = logistic_cifar.score(X_cifar_test_pca, y_cifar_test.ravel())
print(f"CIFAR10 Logistic Regression Accuracy (Standard PCA): {accuracy_cifar:.4f}")

# Train Logistic Regression for MNIST
# Initialize Logistic Regression model with an increased max_iter for convergence.
logistic_mnist = LogisticRegression(max_iter=10000, random_state=42)
# Fit the model using MNIST PCA training features and corresponding labels.
logistic_mnist.fit(X_mnist_train_pca, y_mnist_train.ravel())

# Evaluate MNIST classifier
# Calculate accuracy on the PCA-transformed MNIST test set.
accuracy_mnist = logistic_mnist.score(X_mnist_test_pca, y_mnist_test.ravel())
print(f"MNIST Logistic Regression Accuracy (Standard PCA): {accuracy_mnist:.4f}")

### Plot ROC Curves for CIFAR10 and MNIST (Standard PCA)

In [ ]:
n_classes_cifar = 10
y_cifar_test_bin = label_binarize(y_cifar_test, classes=range(n_classes_cifar))
y_score_cifar = logistic_cifar.predict_proba(X_cifar_test_pca)

# Compute ROC curve and ROC area for each class
fpr = dict()
tpr = dict()
roc_auc = dict()
for i in range(n_classes_cifar):
    fpr[i], tpr[i], _ = roc_curve(y_cifar_test_bin[:, i], y_score_cifar[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Plot ROC curves for CIFAR10
plt.figure(figsize=(10, 8))
for i in range(n_classes_cifar):
    # Use the defined class names for the legend labels
    plt.plot(fpr[i], tpr[i], label=f'{cifar10_class_names[i]} (area = {roc_auc[i]:.2f})')
plt.plot([0, 1], [0, 1], 'k--', label='Chance')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) for CIFAR10 (Standard PCA)')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

In [ ]:
n_classes_mnist = 10
# Binarize the true labels for ROC curve computation
y_mnist_test_bin = label_binarize(y_mnist_test, classes=range(n_classes_mnist))
# Predict probabilities for each class on the PCA-transformed test data
y_score_mnist = logistic_mnist.predict_proba(X_mnist_test_pca)

# Compute ROC curve and ROC area for each class
fpr = dict()
tpr = dict()
roc_auc = dict()
for i in range(n_classes_mnist):
    # Calculate False Positive Rate (fpr), True Positive Rate (tpr), and thresholds for each class
    fpr[i], tpr[i], _ = roc_curve(y_mnist_test_bin[:, i], y_score_mnist[:, i])
    # Calculate the Area Under the Curve (AUC) for each class
    roc_auc[i] = auc(fpr[i], tpr[i])

# Plot ROC curves for MNIST
plt.figure(figsize=(10, 8))
for i in range(n_classes_mnist):
    # Plot each class's ROC curve, using generic class numbers for labels
    plt.plot(fpr[i], tpr[i], label=f'Class {i} (area = {roc_auc[i]:.2f})')
plt.plot([0, 1], [0, 1], 'k--', label='Chance') # Plot the diagonal chance line
plt.xlim([0.0, 1.0]) # Set x-axis limits
plt.ylim([0.0, 1.05]) # Set y-axis limits
plt.xlabel('False Positive Rate') # Label x-axis
plt.ylabel('True Positive Rate') # Label y-axis
plt.title('Receiver Operating Characteristic (ROC) for MNIST (Standard PCA)') # Set plot title
plt.legend(loc="lower right") # Display legend
plt.grid(True) # Add grid
plt.show()

### Randomized PCA

In [ ]:
from sklearn.decomposition import PCA

# Apply Randomized PCA for CIFAR10
# Initialize Randomized PCA with 30 components, 'randomized' svd_solver, and a random state.
# `n_components=30` reduces dimensionality, and `svd_solver='randomized'` uses a probabilistic
# algorithm for faster computation on large datasets.
rpca_cifar = PCA(n_components=30, svd_solver='randomized', random_state=42)
# Fit Randomized PCA on the mean-centered CIFAR10 training data and transform it.
X_cifar_train_rpca = rpca_cifar.fit_transform(X_cifar_train_centered)

# Transform CIFAR10 test data using the Randomized PCA model fitted on the training data.
# It's important to use the *same* fitted model to transform test data to maintain consistency.
X_cifar_test_rpca = rpca_cifar.transform(X_cifar_test_centered)

print(f"CIFAR10 Randomized PCA training features shape: {X_cifar_train_rpca.shape}")
print(f"CIFAR10 Randomized PCA test features shape: {X_cifar_test_rpca.shape}")

# Apply Randomized PCA for MNIST
# Initialize Randomized PCA for MNIST with the same parameters.
rpca_mnist = PCA(n_components=30, svd_solver='randomized', random_state=42)
# Fit Randomized PCA on the mean-centered MNIST training data and transform it.
X_mnist_train_rpca = rpca_mnist.fit_transform(X_mnist_train_centered)

# Transform MNIST test data using the Randomized PCA model.
X_mnist_test_rpca = rpca_mnist.transform(X_mnist_test_centered)

print(f"MNIST Randomized PCA training features shape: {X_mnist_train_rpca.shape}")
print(f"MNIST Randomized PCA test features shape: {X_mnist_test_rpca.shape}")

### Train Logistic Regression Classifiers using Randomized PCA features

In [ ]:
from sklearn.linear_model import LogisticRegression

# Train Logistic Regression for CIFAR10 (Randomized PCA)
# Initialize Logistic Regression model with an increased max_iter for convergence.
# random_state ensures reproducibility.
logistic_cifar_rpca = LogisticRegression(max_iter=10000, random_state=42)
# Fit the model using CIFAR10 Randomized PCA training features and corresponding labels.
logistic_cifar_rpca.fit(X_cifar_train_rpca, y_cifar_train.ravel())

# Evaluate CIFAR10 classifier (Randomized PCA)
# Calculate accuracy on the Randomized PCA-transformed CIFAR10 test set.
accuracy_cifar_rpca = logistic_cifar_rpca.score(X_cifar_test_rpca, y_cifar_test.ravel())
print(f"CIFAR10 Logistic Regression Accuracy (Randomized PCA): {accuracy_cifar_rpca:.4f}")

# Train Logistic Regression for MNIST (Randomized PCA)
# Initialize Logistic Regression model with an increased max_iter for convergence.
logistic_mnist_rpca = LogisticRegression(max_iter=10000, random_state=42)
# Fit the model using MNIST Randomized PCA training features and corresponding labels.
logistic_mnist_rpca.fit(X_mnist_train_rpca, y_mnist_train.ravel())

# Evaluate MNIST classifier (Randomized PCA)
# Calculate accuracy on the Randomized PCA-transformed MNIST test set.
accuracy_mnist_rpca = logistic_mnist_rpca.score(X_mnist_test_rpca, y_mnist_test.ravel())
print(f"MNIST Logistic Regression Accuracy (Randomized PCA): {accuracy_mnist_rpca:.4f}")

### Plot ROC Curves for CIFAR10 and MNIST (Randomized PCA)

In [ ]:
n_classes_cifar = 10
y_cifar_test_bin = label_binarize(y_cifar_test, classes=range(n_classes_cifar))
y_score_cifar_rpca = logistic_cifar_rpca.predict_proba(X_cifar_test_rpca)

# Compute ROC curve and ROC area for each class
fpr = dict()
tpr = dict()
roc_auc = dict()
for i in range(n_classes_cifar):
    fpr[i], tpr[i], _ = roc_curve(y_cifar_test_bin[:, i], y_score_cifar_rpca[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Plot ROC curves for CIFAR10 (Randomized PCA)
plt.figure(figsize=(10, 8))
for i in range(n_classes_cifar):
    # Use the defined class names for the legend labels
    plt.plot(fpr[i], tpr[i], label=f'{cifar10_class_names[i]} (area = {roc_auc[i]:.2f})')
plt.plot([0, 1], [0, 1], 'k--', label='Chance')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) for CIFAR10 (Randomized PCA)')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

In [ ]:
n_classes_mnist = 10
# Binarize the true labels for ROC curve computation
y_mnist_test_bin = label_binarize(y_mnist_test, classes=range(n_classes_mnist))
# Predict probabilities for each class on the Randomized PCA-transformed test data
y_score_mnist_rpca = logistic_mnist_rpca.predict_proba(X_mnist_test_rpca)

# Compute ROC curve and ROC area for each class
fpr = dict()
tpr = dict()
roc_auc = dict()
for i in range(n_classes_mnist):
    # Calculate False Positive Rate (fpr), True Positive Rate (tpr), and thresholds for each class
    fpr[i], tpr[i], _ = roc_curve(y_mnist_test_bin[:, i], y_score_mnist_rpca[:, i])
    # Calculate the Area Under the Curve (AUC) for each class
    roc_auc[i] = auc(fpr[i], tpr[i])

# Plot ROC curves for MNIST (Randomized PCA)
plt.figure(figsize=(10, 8))
for i in range(n_classes_mnist):
    # Plot each class's ROC curve, using generic class numbers for labels
    plt.plot(fpr[i], tpr[i], label=f'Class {i} (area = {roc_auc[i]:.2f})')
plt.plot([0, 1], [0, 1], 'k--', label='Chance') # Plot the diagonal chance line
plt.xlim([0.0, 1.0]) # Set x-axis limits
plt.ylim([0.0, 1.05]) # Set y-axis limits
plt.xlabel('False Positive Rate') # Label x-axis
plt.ylabel('True Positive Rate') # Label y-axis
plt.title('Receiver Operating Characteristic (ROC) for MNIST (Randomized PCA)') # Set plot title
plt.legend(loc="lower right") # Display legend
plt.grid(True) # Add grid
plt.show()

### Reconstruct test images and calculate SNR (Standard vs. Randomized PCA)

In [ ]:
def calculate_snr(original, reconstructed):
    # Calculate the noise as the difference between original and reconstructed signals
    noise = original - reconstructed
    # Calculate the power of the original signal
    power_signal = np.mean(original**2)
    # Calculate the power of the noise
    power_noise = np.mean(noise**2)
    # If there is no noise, SNR is infinite
    if power_noise == 0:
        return np.inf
    # Calculate SNR in decibels (dB)
    return 10 * np.log10(power_signal / power_noise)

# Reconstruct CIFAR10 test images (Standard PCA)
# inverse_transform projects the PCA-transformed data back to the original feature space.
# The mean (subtracted during centering) is added back to get closer to the original pixel values.
X_cifar_test_reconstructed_pca = pca_cifar.inverse_transform(X_cifar_test_pca)
X_cifar_test_reconstructed_pca_original_scale = X_cifar_test_reconstructed_pca + mean_cifar

# Reconstruct CIFAR10 test images (Randomized PCA)
# Similar to Standard PCA, inverse_transform and add back the mean.
X_cifar_test_reconstructed_rpca = rpca_cifar.inverse_transform(X_cifar_test_rpca)
X_cifar_test_reconstructed_rpca_original_scale = X_cifar_test_reconstructed_rpca + mean_cifar

# Calculate SNR for CIFAR10
# The SNR is calculated on the mean-centered data for a fair comparison of reconstruction quality
# in the feature space where PCA operates.
snr_cifar_pca = calculate_snr(X_cifar_test_centered, X_cifar_test_reconstructed_pca)
snr_cifar_rpca = calculate_snr(X_cifar_test_centered, X_cifar_test_reconstructed_rpca)
print(f"CIFAR10 Average Reconstruction SNR (Standard PCA): {snr_cifar_pca:.2f} dB")
print(f"CIFAR10 Average Reconstruction SNR (Randomized PCA): {snr_cifar_rpca:.2f} dB")

# Reconstruct MNIST test images (Standard PCA)
# Inverse transform and add back the mean for MNIST data.
X_mnist_test_reconstructed_pca = pca_mnist.inverse_transform(X_mnist_test_pca)
X_mnist_test_reconstructed_pca_original_scale = X_mnist_test_reconstructed_pca + mean_mnist

# Reconstruct MNIST test images (Randomized PCA)
# Inverse transform and add back the mean for MNIST data.
X_mnist_test_reconstructed_rpca = rpca_mnist.inverse_transform(X_mnist_test_rpca)
X_mnist_test_reconstructed_rpca_original_scale = X_mnist_test_reconstructed_rpca + mean_mnist

# Calculate SNR for MNIST
snr_mnist_pca = calculate_snr(X_mnist_test_centered, X_mnist_test_reconstructed_pca)
snr_mnist_rpca = calculate_snr(X_mnist_test_centered, X_mnist_test_reconstructed_rpca)
print(f"MNIST Average Reconstruction SNR (Standard PCA): {snr_mnist_pca:.2f} dB")
print(f"MNIST Average Reconstruction SNR (Randomized PCA): {snr_mnist_rpca:.2f} dB")

### Visualizing Reconstructed Images (Standard PCA) and Original Images

In [ ]:
def plot_reconstructions(original_flat, reconstructed_flat, num_images=5, title_prefix=''):
    """Compare centered original and reconstructed 28 x 28 images."""
    plt.figure(figsize=(10, 4))
    for image_index in range(num_images):
        plt.subplot(2, num_images, image_index + 1)
        plt.imshow(original_flat[image_index].reshape(28, 28), cmap='gray')
        plt.title('Original')
        plt.axis('off')

        plt.subplot(2, num_images, image_index + 1 + num_images)
        plt.imshow(reconstructed_flat[image_index].reshape(28, 28), cmap='gray')
        plt.title('Reconstructed')
        plt.axis('off')

    plt.suptitle(f'{title_prefix} Original vs. Reconstructed Images', fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

# The inputs are mean-centered, so the display does not force the original
# 50-200 intensity limits onto values that now include negative numbers.
plot_reconstructions(X_cifar_test_centered, X_cifar_test_reconstructed_pca, title_prefix='CIFAR10 (Standard PCA)')
plot_reconstructions(X_mnist_test_centered, X_mnist_test_reconstructed_pca, title_prefix='MNIST (Standard PCA)')

### Visualizing Reconstructed Images (Randomized PCA) and Original Images

In [ ]:
def plot_reconstructions_rpca(original_flat, reconstructed_flat, num_images=5, title_prefix=''):
    """Compare centered original and randomized-PCA reconstructions."""
    plt.figure(figsize=(10, 4))
    for image_index in range(num_images):
        plt.subplot(2, num_images, image_index + 1)
        plt.imshow(original_flat[image_index].reshape(28, 28), cmap='gray')
        plt.title('Original')
        plt.axis('off')

        plt.subplot(2, num_images, image_index + 1 + num_images)
        plt.imshow(reconstructed_flat[image_index].reshape(28, 28), cmap='gray')
        plt.title('Reconstructed')
        plt.axis('off')

    plt.suptitle(f'{title_prefix} Original vs. Reconstructed Images', fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

plot_reconstructions_rpca(X_cifar_test_centered, X_cifar_test_reconstructed_rpca, title_prefix='CIFAR10 (Randomized PCA)')
plot_reconstructions_rpca(X_mnist_test_centered, X_mnist_test_reconstructed_rpca, title_prefix='MNIST (Randomized PCA)')

## Task 2: Linear Autoencoder

### Define and Train a Single-Layer Linear Autoencoder with Tied Weights

In [ ]:
from tensorflow.keras import backend as K
from tensorflow.keras.layers import Input, Dense, Lambda
from tensorflow.keras.models import Model
import tensorflow as tf

# Both datasets are prepared as flattened 28 x 28 grayscale images.
input_dim = 28 * 28
latent_dim = 30

class UnitMagnitudeConstraint(tf.keras.constraints.Constraint):
    """Keep every encoder direction at unit L2 norm after each optimizer step."""

    def __call__(self, weights):
        # Dense weights have shape (input_dim, latent_dim). Normalizing each
        # column makes every learned latent direction have norm one.
        column_norms = K.sqrt(K.sum(K.square(weights), axis=0, keepdims=True))
        return weights / (column_norms + K.epsilon())


def build_linear_autoencoder(input_dim=784, latent_dim=30):
    """Build a one-layer linear autoencoder with tied decoder weights."""
    input_img = Input(shape=(input_dim,), name='input_image')

    # A linear Dense layer maps each centered image to a 30-dimensional code.
    encoder_layer = Dense(
        latent_dim,
        activation='linear',
        use_bias=False,
        kernel_constraint=UnitMagnitudeConstraint(),
        name='encoder_weights'
    )
    encoded = encoder_layer(input_img)

    # Tied weights use W^T for reconstruction instead of learning a second
    # decoder matrix. This matches PCA's projection/reconstruction structure.
    decoded = Lambda(
        lambda latent: K.dot(latent, K.transpose(encoder_layer.kernel)),
        name='decoder_output'
    )(encoded)

    autoencoder = Model(input_img, decoded, name='linear_tied_autoencoder')
    autoencoder.compile(optimizer='adam', loss='mean_squared_error')
    return autoencoder


# Use independent models so MNIST training cannot overwrite CIFAR10 weights.
linear_autoencoder_cifar = build_linear_autoencoder(input_dim, latent_dim)
linear_autoencoder_mnist = build_linear_autoencoder(input_dim, latent_dim)

print('CIFAR10 linear autoencoder:')
linear_autoencoder_cifar.summary()
print('MNIST linear autoencoder:')
linear_autoencoder_mnist.summary()

# Centered, flattened images are used for both training and PCA comparison.
X_cifar_train_ae = X_cifar_train_centered
X_cifar_val_ae = X_cifar_val_centered
X_cifar_test_ae = X_cifar_test_centered
X_mnist_train_ae = X_mnist_train_centered
X_mnist_val_ae = X_mnist_val_centered
X_mnist_test_ae = X_mnist_test_centered

print('Linear autoencoders and centered data are ready.')

#### Train Autoencoder for CIFAR10

In [ ]:
# Train only the CIFAR10-specific model on centered CIFAR10 images.
# The validation set monitors generalization during training; it is never used
# to update the weights. A moderate batch size keeps Colab memory usage stable.
history_cifar_ae = linear_autoencoder_cifar.fit(
    X_cifar_train_ae,
    X_cifar_train_ae,
    epochs=50,
    batch_size=256,
    shuffle=True,
    validation_data=(X_cifar_val_ae, X_cifar_val_ae),
    verbose=1
)

# Plot both curves against the same epoch axis. A small validation loss means
# the tied linear projection reconstructs unseen CIFAR10 images well.
plt.figure(figsize=(10, 6))
plt.plot(history_cifar_ae.history['loss'], label='Train loss')
plt.plot(history_cifar_ae.history['val_loss'], label='Validation loss')
plt.title('CIFAR10 Linear Autoencoder Loss')
plt.ylabel('Mean squared error')
plt.xlabel('Epoch')
plt.legend(loc='upper right')
plt.grid(True)
plt.show()

#### Train Autoencoder for MNIST

In [ ]:
# Train the independent MNIST-specific model. It starts with fresh weights,
# so the learned MNIST representation is not influenced by CIFAR10 training.
history_mnist_ae = linear_autoencoder_mnist.fit(
    X_mnist_train_ae,
    X_mnist_train_ae,
    epochs=50,
    batch_size=256,
    shuffle=True,
    validation_data=(X_mnist_val_ae, X_mnist_val_ae),
    verbose=1
)

# The training and validation curves make optimization or overfitting visible.
# Use the final values as a compact numerical summary of reconstruction quality.
plt.figure(figsize=(10, 6))
plt.plot(history_mnist_ae.history['loss'], label='Train loss')
plt.plot(history_mnist_ae.history['val_loss'], label='Validation loss')
plt.title('MNIST Linear Autoencoder Loss')
plt.ylabel('Mean squared error')
plt.xlabel('Epoch')
plt.legend(loc='upper right')
plt.grid(True)
plt.show()

print(f"Final MNIST train MSE: {history_mnist_ae.history['loss'][-1]:.6f}")
print(f"Final MNIST validation MSE: {history_mnist_ae.history['val_loss'][-1]:.6f}")

### Compare Autoencoder Representation with PCA

#### Qualitative Comparison: Visualize PCA Eigenvectors and Autoencoder Weight Vectors

In [ ]:
def plot_components(components, title, num_components=30):
    """Display learned directions as 28 x 28 grayscale images."""
    plt.figure(figsize=(15, 6))
    for component_index in range(num_components):
        plt.subplot(3, 10, component_index + 1)
        plt.imshow(components[component_index].reshape(28, 28), cmap='gray', vmin=-1, vmax=1)
        plt.axis('off')
    plt.suptitle(title, fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

# PCA components are orthonormal directions learned from each training set.
pca_components_cifar = pca_cifar.components_
pca_components_mnist = pca_mnist.components_

# Each AE encoder has shape (784, 30); transpose it so each direction has the
# same shape (30, 784) as a PCA component. Use the matching dataset model.
ae_encoder_weights_cifar = linear_autoencoder_cifar.get_layer('encoder_weights').get_weights()[0].T
ae_encoder_weights_mnist = linear_autoencoder_mnist.get_layer('encoder_weights').get_weights()[0].T

print('CIFAR10: PCA components versus tied linear-AE encoder directions')
plot_components(pca_components_cifar, 'CIFAR10 PCA Components')
plot_components(ae_encoder_weights_cifar, 'CIFAR10 Linear-AE Encoder Directions')

print('MNIST: PCA components versus tied linear-AE encoder directions')
plot_components(pca_components_mnist, 'MNIST PCA Components')
plot_components(ae_encoder_weights_mnist, 'MNIST Linear-AE Encoder Directions')

#### Quantitative Comparison: Subspace-Comparison Metric

In [ ]:
from scipy.linalg import orth


def subspace_angle(pca_directions, ae_directions):
    """Return the mean principal angle, in degrees, between two subspaces."""
    # orth(A.T) converts rows of A into an orthonormal basis. The singular
    # values of Q_pca.T @ Q_ae are cosines of the principal angles.
    pca_basis = orth(pca_directions.T)
    ae_basis = orth(ae_directions.T)
    singular_values = np.linalg.svd(pca_basis.T @ ae_basis, compute_uv=False)
    principal_angles = np.arccos(np.clip(singular_values, -1.0, 1.0))
    return np.degrees(principal_angles).mean()


# Compare matching dataset subspaces. Smaller angles indicate that the AE and
# PCA learn more similar 30-dimensional directions; signs and ordering of
# individual directions do not affect this subspace-level metric.
cifar_angle = subspace_angle(pca_components_cifar, ae_encoder_weights_cifar)
mnist_angle = subspace_angle(pca_components_mnist, ae_encoder_weights_mnist)

print(f"CIFAR10 mean PCA/linear-AE principal angle: {cifar_angle:.2f} degrees")
print(f"MNIST mean PCA/linear-AE principal angle: {mnist_angle:.2f} degrees")

### Train Logistic Regression Classifiers using Autoencoder features

In [ ]:
# Build one feature extractor per trained AE. Each extractor returns the
# 30-dimensional linear code produced by its matching encoder.
cifar_feature_extractor = Model(
    inputs=linear_autoencoder_cifar.input,
    outputs=linear_autoencoder_cifar.get_layer('encoder_weights').output
)
mnist_feature_extractor = Model(
    inputs=linear_autoencoder_mnist.input,
    outputs=linear_autoencoder_mnist.get_layer('encoder_weights').output
)

X_cifar_train_ae_features = cifar_feature_extractor.predict(X_cifar_train_ae, verbose=0)
X_cifar_test_ae_features = cifar_feature_extractor.predict(X_cifar_test_ae, verbose=0)
X_mnist_train_ae_features = mnist_feature_extractor.predict(X_mnist_train_ae, verbose=0)
X_mnist_test_ae_features = mnist_feature_extractor.predict(X_mnist_test_ae, verbose=0)

# Logistic regression evaluates how useful each learned representation is for
# classification. The classifier is fitted on training codes and evaluated on
# held-out test codes; max_iter=10000 gives the optimizer enough iterations to
# converge on the multiclass problem.
logistic_cifar_ae = LogisticRegression(max_iter=10000, random_state=42)
logistic_cifar_ae.fit(X_cifar_train_ae_features, y_cifar_train.ravel())
accuracy_cifar_ae = logistic_cifar_ae.score(X_cifar_test_ae_features, y_cifar_test.ravel())

logistic_mnist_ae = LogisticRegression(max_iter=10000, random_state=42)
logistic_mnist_ae.fit(X_mnist_train_ae_features, y_mnist_train.ravel())
accuracy_mnist_ae = logistic_mnist_ae.score(X_mnist_test_ae_features, y_mnist_test.ravel())

print(f"CIFAR10 Logistic Regression Accuracy (linear-AE features): {accuracy_cifar_ae:.4f}")
print(f"MNIST Logistic Regression Accuracy (linear-AE features): {accuracy_mnist_ae:.4f}")

## Task 3: Nonlinear, Deep Dense, and Convolutional Autoencoders

### 3.1 Nonlinear Shallow Autoencoder

In [ ]:
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model

# Each input is a centered, flattened 28 x 28 grayscale image (784 values).
# The bottleneck has 30 values so this model can be compared directly with PCA.
input_dim = 28 * 28
latent_dim = 30

# The shallow model intentionally contains only one nonlinear encoder layer.
# ReLU keeps positive activations and gives the representation more flexibility
# than the linear autoencoder, while the linear decoder can reconstruct the
# centered pixel values without restricting them to [0, 1].
input_img = Input(shape=(input_dim,), name='shallow_input')
encoded = Dense(latent_dim, activation='relu', name='encoder_output')(input_img)
decoded = Dense(input_dim, activation='linear', name='decoder_output')(encoded)

# The model learns to reproduce its input, so no class labels are required.
# Adam adapts the learning rate during optimization and MSE penalizes pixelwise
# reconstruction errors. On Colab, place model creation and fit inside a
# tf.distribute.TPUStrategy scope when a TPU runtime is selected for faster
# training; the same model also runs on CPU or GPU.
shallow_nonlinear_ae = Model(input_img, decoded, name='shallow_nonlinear_autoencoder')
shallow_nonlinear_ae.compile(optimizer='adam', loss='mean_squared_error')

print('Shallow Nonlinear Autoencoder Architecture:')
shallow_nonlinear_ae.summary()

#### Train Shallow Nonlinear Autoencoder for CIFAR10

In [ ]:
# Train the shallow nonlinear autoencoder on CIFAR10. The input is also the
# target because an autoencoder learns without labels. Validation data is used
# only to monitor generalization. A TPU runtime can shorten these epochs when
# the model is created inside a tf.distribute.TPUStrategy scope.
history_cifar_shallow_nl = shallow_nonlinear_ae.fit(
    X_cifar_train_ae,
    X_cifar_train_ae,
    epochs=50,
    batch_size=256,
    shuffle=True,
    validation_data=(X_cifar_val_ae, X_cifar_val_ae),
    verbose=1
)

# Plot training and validation MSE to check convergence and overfitting.
plt.figure(figsize=(10, 6))
plt.plot(history_cifar_shallow_nl.history['loss'], label='Train loss')
plt.plot(history_cifar_shallow_nl.history['val_loss'], label='Validation loss')
plt.title('CIFAR10 Shallow Nonlinear Autoencoder Loss')
plt.ylabel('Mean squared error')
plt.xlabel('Epoch')
plt.legend(loc='upper right')
plt.grid(True)
plt.show()

# SNR summarizes reconstruction quality in the same centered feature space.
cifar_reconstructed_train = shallow_nonlinear_ae.predict(X_cifar_train_ae, verbose=0)
cifar_reconstructed_test = shallow_nonlinear_ae.predict(X_cifar_test_ae, verbose=0)
snr_cifar_shallow_nl_train = calculate_snr(X_cifar_train_ae, cifar_reconstructed_train)
snr_cifar_shallow_nl_test = calculate_snr(X_cifar_test_ae, cifar_reconstructed_test)

print(f"CIFAR10 final train MSE: {history_cifar_shallow_nl.history['loss'][-1]:.6f}")
print(f"CIFAR10 final validation MSE: {history_cifar_shallow_nl.history['val_loss'][-1]:.6f}")
print(f"CIFAR10 train SNR: {snr_cifar_shallow_nl_train:.2f} dB")
print(f"CIFAR10 test SNR: {snr_cifar_shallow_nl_test:.2f} dB")

#### Train Shallow Nonlinear Autoencoder for MNIST

In [ ]:
# Train the same shallow architecture independently on MNIST. Reusing the
# architecture is intentional, but this fit call uses MNIST data and labels
# are not needed for reconstruction. TPU execution can accelerate training in
# Colab when the model is built under a TPU strategy scope.
history_mnist_shallow_nl = shallow_nonlinear_ae.fit(
    X_mnist_train_ae,
    X_mnist_train_ae,
    epochs=50,
    batch_size=256,
    shuffle=True,
    validation_data=(X_mnist_val_ae, X_mnist_val_ae),
    verbose=1
)

# Compare training and validation MSE across epochs.
plt.figure(figsize=(10, 6))
plt.plot(history_mnist_shallow_nl.history['loss'], label='Train loss')
plt.plot(history_mnist_shallow_nl.history['val_loss'], label='Validation loss')
plt.title('MNIST Shallow Nonlinear Autoencoder Loss')
plt.ylabel('Mean squared error')
plt.xlabel('Epoch')
plt.legend(loc='upper right')
plt.grid(True)
plt.show()

mnist_reconstructed_train = shallow_nonlinear_ae.predict(X_mnist_train_ae, verbose=0)
mnist_reconstructed_test = shallow_nonlinear_ae.predict(X_mnist_test_ae, verbose=0)
snr_mnist_shallow_nl_train = calculate_snr(X_mnist_train_ae, mnist_reconstructed_train)
snr_mnist_shallow_nl_test = calculate_snr(X_mnist_test_ae, mnist_reconstructed_test)

print(f"MNIST final train MSE: {history_mnist_shallow_nl.history['loss'][-1]:.6f}")
print(f"MNIST final validation MSE: {history_mnist_shallow_nl.history['val_loss'][-1]:.6f}")
print(f"MNIST train SNR: {snr_mnist_shallow_nl_train:.2f} dB")
print(f"MNIST test SNR: {snr_mnist_shallow_nl_test:.2f} dB")

### 3.2 Symmetric Deep Dense Autoencoder

In [ ]:
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model

# The deep dense autoencoder keeps the same 784-input and 30-dimensional
# bottleneck as PCA, but adds nonlinear capacity on both sides of the code.
input_dim = 28 * 28
latent_dim = 30
hidden_1_units = 256
hidden_2_units = 128

input_img = Input(shape=(input_dim,), name='deep_dense_input')

# The encoder progressively compresses 784 -> 256 -> 128 -> 30 values.
# ReLU activations model nonlinear structure while keeping optimization stable.
encoded = Dense(hidden_1_units, activation='relu', name='encoder_dense_256')(input_img)
encoded = Dense(hidden_2_units, activation='relu', name='encoder_dense_128')(encoded)
encoded = Dense(latent_dim, activation='relu', name='encoder_output')(encoded)

# The decoder mirrors the encoder: 30 -> 128 -> 256 -> 784. A linear output
# is appropriate because the targets are mean-centered pixel intensities.
decoded = Dense(hidden_2_units, activation='relu', name='decoder_dense_128')(encoded)
decoded = Dense(hidden_1_units, activation='relu', name='decoder_dense_256')(decoded)
decoded = Dense(input_dim, activation='linear', name='decoder_output')(decoded)

# This is an unsupervised reconstruction task. Adam minimizes MSE between each
# input and reconstruction. A Colab TPU can accelerate the repeated dense
# matrix operations; use a TPU strategy scope when configuring the model there.
deep_dense_ae = Model(input_img, decoded, name='deep_dense_autoencoder')
deep_dense_ae.compile(optimizer='adam', loss='mean_squared_error')

print('Deep Dense Autoencoder Architecture:')
deep_dense_ae.summary()

#### Train Deep Dense Autoencoder for CIFAR10

In [ ]:
# Train the symmetric deep dense autoencoder on centered CIFAR10 vectors.
# The validation split monitors generalization while shuffle=True changes the
# minibatch order each epoch. TPU hardware can speed up the dense operations
# in Colab when the model is created inside a tf.distribute.TPUStrategy scope.
history_cifar_deep_dense = deep_dense_ae.fit(
    X_cifar_train_ae,
    X_cifar_train_ae,
    epochs=50,
    batch_size=256,
    shuffle=True,
    validation_data=(X_cifar_val_ae, X_cifar_val_ae),
    verbose=1
)

# Divergence between the curves can indicate overfitting; decreasing curves
# indicate that the compressed representation is learning useful structure.
plt.figure(figsize=(10, 6))
plt.plot(history_cifar_deep_dense.history['loss'], label='Train loss')
plt.plot(history_cifar_deep_dense.history['val_loss'], label='Validation loss')
plt.title('CIFAR10 Deep Dense Autoencoder Loss')
plt.ylabel('Mean squared error')
plt.xlabel('Epoch')
plt.legend(loc='upper right')
plt.grid(True)
plt.show()

cifar_reconstructed_train = deep_dense_ae.predict(X_cifar_train_ae, verbose=0)
cifar_reconstructed_test = deep_dense_ae.predict(X_cifar_test_ae, verbose=0)
snr_cifar_deep_dense_train = calculate_snr(X_cifar_train_ae, cifar_reconstructed_train)
snr_cifar_deep_dense_test = calculate_snr(X_cifar_test_ae, cifar_reconstructed_test)

print(f"CIFAR10 final train MSE: {history_cifar_deep_dense.history['loss'][-1]:.6f}")
print(f"CIFAR10 final validation MSE: {history_cifar_deep_dense.history['val_loss'][-1]:.6f}")
print(f"CIFAR10 train SNR: {snr_cifar_deep_dense_train:.2f} dB")
print(f"CIFAR10 test SNR: {snr_cifar_deep_dense_test:.2f} dB")

#### Train Deep Dense Autoencoder for MNIST

In [ ]:
# Train the symmetric deep dense autoencoder on MNIST using the same
# unsupervised reconstruction objective as CIFAR10. A TPU runtime in Colab can
# accelerate this fit when the model is created under a TPU strategy scope.
history_mnist_deep_dense = deep_dense_ae.fit(
    X_mnist_train_ae,
    X_mnist_train_ae,
    epochs=50,
    batch_size=256,
    shuffle=True,
    validation_data=(X_mnist_val_ae, X_mnist_val_ae),
    verbose=1
)

plt.figure(figsize=(10, 6))
plt.plot(history_mnist_deep_dense.history['loss'], label='Train loss')
plt.plot(history_mnist_deep_dense.history['val_loss'], label='Validation loss')
plt.title('MNIST Deep Dense Autoencoder Loss')
plt.ylabel('Mean squared error')
plt.xlabel('Epoch')
plt.legend(loc='upper right')
plt.grid(True)
plt.show()

mnist_reconstructed_train = deep_dense_ae.predict(X_mnist_train_ae, verbose=0)
mnist_reconstructed_test = deep_dense_ae.predict(X_mnist_test_ae, verbose=0)
snr_mnist_deep_dense_train = calculate_snr(X_mnist_train_ae, mnist_reconstructed_train)
snr_mnist_deep_dense_test = calculate_snr(X_mnist_test_ae, mnist_reconstructed_test)

print(f"MNIST final train MSE: {history_mnist_deep_dense.history['loss'][-1]:.6f}")
print(f"MNIST final validation MSE: {history_mnist_deep_dense.history['val_loss'][-1]:.6f}")
print(f"MNIST train SNR: {snr_mnist_deep_dense_train:.2f} dB")
print(f"MNIST test SNR: {snr_mnist_deep_dense_test:.2f} dB")

### 3.3 Deep Convolutional Autoencoder

In [ ]:
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, Flatten, Reshape, Dense
from tensorflow.keras.models import Model

# The CNN receives unflattened 28 x 28 x 1 images so convolution can exploit
# local spatial patterns such as edges and strokes. The latent size remains 30
# for comparison with the dense autoencoders and PCA.
img_rows, img_cols, img_channels = 28, 28, 1
latent_dim = 30
input_img = Input(shape=(img_rows, img_cols, img_channels), name='conv_input')

# Encoder: same-padding convolutions preserve spatial detail, while pooling
# halves the resolution and increases the channel count: 28x28x1 -> 14x14x32
# -> 7x7x64. Flattening then produces 3136 values for the bottleneck layer.
x = Conv2D(32, (3, 3), activation='relu', padding='same', name='encoder_conv_32')(input_img)
x = MaxPooling2D((2, 2), padding='same', name='encoder_pool_14')(x)
x = Conv2D(64, (3, 3), activation='relu', padding='same', name='encoder_conv_64')(x)
x = MaxPooling2D((2, 2), padding='same', name='encoder_pool_7')(x)
flattened = Flatten(name='encoder_flatten')(x)
bottleneck = Dense(latent_dim, activation='relu', name='encoder_output')(flattened)

# Decoder: expand the code back to 7x7x64, then reverse the encoder using
# convolutions and nearest-neighbor upsampling: 7x7 -> 14x14 -> 28x28.
x = Dense(7 * 7 * 64, activation='relu', name='decoder_expand')(bottleneck)
x = Reshape((7, 7, 64), name='decoder_reshape')(x)
x = Conv2D(64, (3, 3), activation='relu', padding='same', name='decoder_conv_64')(x)
x = UpSampling2D((2, 2), name='decoder_upsample_14')(x)
x = Conv2D(32, (3, 3), activation='relu', padding='same', name='decoder_conv_32')(x)
x = UpSampling2D((2, 2), name='decoder_upsample_28')(x)
decoded = Conv2D(img_channels, (3, 3), activation='linear', padding='same', name='decoder_output')(x)

# The CNN is trained without labels to minimize reconstruction MSE. Colab's
# TPU can substantially speed up these convolution-heavy epochs; configure a
# tf.distribute.TPUStrategy before creating the model when a TPU is available.
conv_ae = Model(input_img, decoded, name='deep_convolutional_autoencoder')
conv_ae.compile(optimizer='adam', loss='mean_squared_error')

print('Deep Convolutional Autoencoder Architecture:')
conv_ae.summary()

# Convolutional layers require image-shaped arrays. The mean is computed from
# each dataset's training split, then broadcast over height, width, and channel.
X_cifar_train_conv_ae = X_cifar_train - mean_cifar.reshape(1, 28, 28, 1)
X_cifar_val_conv_ae = X_cifar_val - mean_cifar.reshape(1, 28, 28, 1)
X_cifar_test_conv_ae = X_cifar_test - mean_cifar.reshape(1, 28, 28, 1)
X_mnist_train_conv_ae = X_mnist_train - mean_mnist.reshape(1, 28, 28, 1)
X_mnist_val_conv_ae = X_mnist_val - mean_mnist.reshape(1, 28, 28, 1)
X_mnist_test_conv_ae = X_mnist_test - mean_mnist.reshape(1, 28, 28, 1)

print('Convolutional autoencoder and image-shaped centered data are ready.')

#### Train Deep Convolutional Autoencoder for CIFAR10

In [ ]:
# CNNs expect image-shaped tensors, so use the centered CIFAR10 arrays created
# in the architecture cell. The input is also the target for reconstruction.
# On Colab, a TPU can substantially accelerate these convolution-heavy epochs
# when the model is created inside a tf.distribute.TPUStrategy scope.
history_cifar_conv = conv_ae.fit(
    X_cifar_train_conv_ae,
    X_cifar_train_conv_ae,
    epochs=50,
    batch_size=256,
    shuffle=True,
    validation_data=(X_cifar_val_conv_ae, X_cifar_val_conv_ae),
    verbose=1
)

# Training and validation MSE show convergence and expose overfitting.
plt.figure(figsize=(10, 6))
plt.plot(history_cifar_conv.history['loss'], label='Train loss')
plt.plot(history_cifar_conv.history['val_loss'], label='Validation loss')
plt.title('CIFAR10 Deep Convolutional Autoencoder Loss')
plt.ylabel('Mean squared error')
plt.xlabel('Epoch')
plt.legend(loc='upper right')
plt.grid(True)
plt.show()

# Flatten only for the scalar SNR calculation; the CNN itself receives 4-D data.
cifar_reconstructed_train = conv_ae.predict(X_cifar_train_conv_ae, verbose=0)
cifar_reconstructed_test = conv_ae.predict(X_cifar_test_conv_ae, verbose=0)
snr_cifar_conv_train = calculate_snr(
    X_cifar_train_conv_ae.reshape(X_cifar_train_conv_ae.shape[0], -1),
    cifar_reconstructed_train.reshape(cifar_reconstructed_train.shape[0], -1)
)
snr_cifar_conv_test = calculate_snr(
    X_cifar_test_conv_ae.reshape(X_cifar_test_conv_ae.shape[0], -1),
    cifar_reconstructed_test.reshape(cifar_reconstructed_test.shape[0], -1)
)

print(f"CIFAR10 final train MSE: {history_cifar_conv.history['loss'][-1]:.6f}")
print(f"CIFAR10 final validation MSE: {history_cifar_conv.history['val_loss'][-1]:.6f}")
print(f"CIFAR10 train SNR: {snr_cifar_conv_train:.2f} dB")
print(f"CIFAR10 test SNR: {snr_cifar_conv_test:.2f} dB")

#### Train Deep Convolutional Autoencoder for MNIST

In [ ]:
# Train the CNN on the correctly shaped MNIST tensors. In particular,
# X_mnist_train_conv_ae is the input and reconstruction target, while
# X_mnist_val_conv_ae is reserved for validation. This confirms the MNIST path
# is separate from the flattened dense-autoencoder variables. A Colab TPU can
# speed up the convolutional fit when the model uses a TPU strategy scope.
history_mnist_conv = conv_ae.fit(
    X_mnist_train_conv_ae,
    X_mnist_train_conv_ae,
    epochs=50,
    batch_size=256,
    shuffle=True,
    validation_data=(X_mnist_val_conv_ae, X_mnist_val_conv_ae),
    verbose=1
)

# Plot both MSE curves so convergence and any train/validation gap are visible.
plt.figure(figsize=(10, 6))
plt.plot(history_mnist_conv.history['loss'], label='Train loss')
plt.plot(history_mnist_conv.history['val_loss'], label='Validation loss')
plt.title('MNIST Deep Convolutional Autoencoder Loss')
plt.ylabel('Mean squared error')
plt.xlabel('Epoch')
plt.legend(loc='upper right')
plt.grid(True)
plt.show()

# Keep the CNN output 4-D during prediction. Flatten both arrays only because
# SNR is a scalar signal-to-reconstruction-noise calculation.
mnist_reconstructed_train = conv_ae.predict(X_mnist_train_conv_ae, verbose=0)
mnist_reconstructed_test = conv_ae.predict(X_mnist_test_conv_ae, verbose=0)
snr_mnist_conv_train = calculate_snr(
    X_mnist_train_conv_ae.reshape(X_mnist_train_conv_ae.shape[0], -1),
    mnist_reconstructed_train.reshape(mnist_reconstructed_train.shape[0], -1)
)
snr_mnist_conv_test = calculate_snr(
    X_mnist_test_conv_ae.reshape(X_mnist_test_conv_ae.shape[0], -1),
    mnist_reconstructed_test.reshape(mnist_reconstructed_test.shape[0], -1)
)

print(f"MNIST final train MSE: {history_mnist_conv.history['loss'][-1]:.6f}")
print(f"MNIST final validation MSE: {history_mnist_conv.history['val_loss'][-1]:.6f}")
print(f"MNIST train SNR: {snr_mnist_conv_train:.2f} dB")
print(f"MNIST test SNR: {snr_mnist_conv_test:.2f} dB")

## Final Comparison and Summary

This table compares reconstruction quality across all representation-learning models. Lower MSE and higher SNR indicate better reconstruction. Classification accuracy is reported for the PCA and linear-autoencoder features evaluated with logistic regression; the nonlinear and deep autoencoders are compared by reconstruction quality in this notebook.

In [ ]:
import pandas as pd

# Convert each test SNR to the corresponding mean-squared reconstruction error.
# This keeps all models comparable even though their test reconstructions are
# produced in separate training cells.
def mse_from_snr(original, snr_db):
    signal_power = np.mean(original ** 2)
    return signal_power / (10 ** (snr_db / 10))

linear_cifar_reconstruction = linear_autoencoder_cifar.predict(X_cifar_test_ae, verbose=0)
linear_mnist_reconstruction = linear_autoencoder_mnist.predict(X_mnist_test_ae, verbose=0)

results = [
    ['CIFAR10', 'Standard PCA', np.mean((X_cifar_test_centered - X_cifar_test_reconstructed_pca) ** 2), snr_cifar_pca, accuracy_cifar],
    ['CIFAR10', 'Randomized PCA', np.mean((X_cifar_test_centered - X_cifar_test_reconstructed_rpca) ** 2), snr_cifar_rpca, accuracy_cifar_rpca],
    ['CIFAR10', 'Linear Autoencoder', np.mean((X_cifar_test_ae - linear_cifar_reconstruction) ** 2), calculate_snr(X_cifar_test_ae, linear_cifar_reconstruction), accuracy_cifar_ae],
    ['CIFAR10', 'Shallow Nonlinear AE', mse_from_snr(X_cifar_test_ae, snr_cifar_shallow_nl_test), snr_cifar_shallow_nl_test, np.nan],
    ['CIFAR10', 'Deep Dense AE', mse_from_snr(X_cifar_test_ae, snr_cifar_deep_dense_test), snr_cifar_deep_dense_test, np.nan],
    ['CIFAR10', 'Deep Convolutional AE', mse_from_snr(X_cifar_test_conv_ae, snr_cifar_conv_test), snr_cifar_conv_test, np.nan],
    ['MNIST', 'Standard PCA', np.mean((X_mnist_test_centered - X_mnist_test_reconstructed_pca) ** 2), snr_mnist_pca, accuracy_mnist],
    ['MNIST', 'Randomized PCA', np.mean((X_mnist_test_centered - X_mnist_test_reconstructed_rpca) ** 2), snr_mnist_rpca, accuracy_mnist_rpca],
    ['MNIST', 'Linear Autoencoder', np.mean((X_mnist_test_ae - linear_mnist_reconstruction) ** 2), calculate_snr(X_mnist_test_ae, linear_mnist_reconstruction), accuracy_mnist_ae],
    ['MNIST', 'Shallow Nonlinear AE', mse_from_snr(X_mnist_test_ae, snr_mnist_shallow_nl_test), snr_mnist_shallow_nl_test, np.nan],
    ['MNIST', 'Deep Dense AE', mse_from_snr(X_mnist_test_ae, snr_mnist_deep_dense_test), snr_mnist_deep_dense_test, np.nan],
    ['MNIST', 'Deep Convolutional AE', mse_from_snr(X_mnist_test_conv_ae, snr_mnist_conv_test), snr_mnist_conv_test, np.nan],
]

comparison_df = pd.DataFrame(
    results,
    columns=['Dataset', 'Model', 'Test MSE', 'Test SNR (dB)', 'Classification Accuracy']
)
comparison_df[['Test MSE', 'Test SNR (dB)', 'Classification Accuracy']] = comparison_df[
    ['Test MSE', 'Test SNR (dB)', 'Classification Accuracy']
].round(4)

display(comparison_df)
print('Lower Test MSE and higher Test SNR indicate better reconstruction.')
print('Classification accuracy is reported for PCA and linear-autoencoder representations.')